In [5]:
using Pkg; Pkg.activate("..")

using Dates
using Statistics
using Interpolations
using ForwardDiff
using NCDatasets
using PythonPlot

using PythonCall

# Py method interprets Array{Union{T,Missing}} as a
# numpy masked array.
# This allows for plotting with missing values.
function PythonCall.Py(a::Array{Union{T,Missing},N}) where {T,N}
    numpy_ma = pyimport("numpy").ma
    numpy_ma.array(coalesce.(a,zero(T)), mask=ismissing.(a))
end

plt = pyimport("matplotlib.pyplot")
cc = pyimport("metpy.plots")
units = pyimport("metpy.units").units

# use a sans family font like Arial, Helvetica, Open Sans
rc("font", family="sans-serif")
rc("font"; Symbol("sans-serif") => ["Arial", "Helvetica", "Open Sans"])
rc("font", size=12)

  Activating project at `~/projects/ASTRAL/soundings`


Python: None

In [ ]:
include(abspath("sonde_profiles.jl"))
profs = all_profiles()[1]            # Revelle 2023, TGT 2024, IMD, Amini
# loads in ~2 min

┌ Warning: CSV: 1 parse problem in <GenericIOBuffer>; first: long_row at data row 1, column 0: expected 11 fields, found 12 (extra fields ignored). Inspect CSV.problems(file); pass on_error=:collect to silence this warning or on_error=:error to throw.
└ @ CSV /home/deszoeks/.julia/packages/CSV/E5gMM/src/core.jl:2984


(NamedTuple[(source = "Revelle", station = "Revelle", time = DateTime("2023-06-17T00:00:00"), lat = 12.057, lon = 67.78558, p = [1008.4, 1007.3, 1006.2, 1005.0, 1003.9, 1002.8, 1001.6, 1000.5, 999.4, 998.3  …  44.5, 44.4, 44.3, 44.3, 44.2, 44.1, 44.1, 44.0, 43.9, 43.8], z = [20.0, 30.0, 40.0, 50.0, 60.0, 70.0, 80.0, 90.0, 100.0, 110.0  …  21460.0, 21470.0, 21480.0, 21490.0, 21500.0, 21510.0, 21520.0, 21530.0, 21540.0, 21550.0], θ = [296.14212870865083, 299.5275505373331, 301.4178000442756, 301.3207614701387, 301.3150691203746, 301.40936724271654, 301.3124809200088, 301.30700918878205, 301.4016254601972, 301.39632711649705  …  514.9965131557885, 516.0570514679644, 516.8760807608036, 516.8760807608036, 517.2095742686615, 517.3003168716309, 517.5440390468355, 517.8794801361325, 518.215902615054, 518.3091141376722], rh = [43.0, 47.0, 52.0, 61.0, 71.0, 76.0, 76.0, 76.0, 76.0, 77.0  …  2.0, 3.0, 4.0, 1.0, 2.0, 2.0, 2.0, 3.0, 1.0, 3.0], u = [0.45532281931361135, 6.166035751283294, 9.413333867

In [20]:
pgrid = 10:10:1000  # hPa grid from 10 to 1000 hPa
prof_interp = Vector{typeof(profs[1])}(length(profs))

MethodError: MethodError: no method matching Vector{@NamedTuple{source::String, station::String, time::DateTime, lat::Float64, lon::Float64, p::Vector{Float64}, z::Vector{Float64}, θ::Vector{Float64}, rh::Vector{Float64}, u::Vector{Float64}, v::Vector{Float64}, z850::Float64, nfree::Int64}}(::Int64)
The type `Vector{@NamedTuple{source::String, station::String, time::DateTime, lat::Float64, lon::Float64, p::Vector{Float64}, z::Vector{Float64}, θ::Vector{Float64}, rh::Vector{Float64}, u::Vector{Float64}, v::Vector{Float64}, z850::Float64, nfree::Int64}}` exists, but no method is defined for this combination of argument types when trying to construct it.

Closest candidates are:
  Vector{T}(!Matched::UndefInitializer, !Matched::Tuple{Int64}) where T
   @ Core boot.jl:661
  Vector{T}(!Matched::UndefInitializer, !Matched::Int64) where T
   @ Core boot.jl:647
  Vector{T}(!Matched::UndefInitializer, !Matched::Integer) where T
   @ Base baseext.jl:23
  ...


In [ ]:
# interpolate mandatory and significant levels to 10-hPa grid
# average if higher resolution than 10 hPa
pgrid = 10:10:1000  # hPa grid from 10 to 1000 hPa
prof_interp = Vector{typeof(profs[1])}(length(profs))
for p in profs
    # interpolate to 10 hPa grid
    itp = interpolate((p.p,), p.T, Gridded(Linear()))
    p.T = itp.(pgrid)
    p.p = pgrid
end

11-element Vector{Float64}:
 1005.0
 1000.0
  974.0
  939.0
  850.0
  700.0
  500.0
  400.0
  300.0
  250.0
  200.0

In [ ]:
# select sondes from May in/adjacent to the Arabian Sea
as = filter(p -> month(p.time) in 5 && p.lon<=77.57, profs[1]) |> collect
